# NB14 — ENIGMA MAG Geochemistry Discovery + Site-Level Correlation (Exploratory)

**Purpose:** Systematic discovery of all geochemistry data in BERDL that can be linked to
the 185 FRC MAGs, followed by correlation of per-Mb 140-KO density with groundwater metal
concentrations.

**Background:** NB11 used only `enigma_coral.ddt_brick0000007`, which matched 29/185 MAGs
across 3 wells. This notebook searches for additional coverage. Isolate genomes (n=2,925)
have no `sample_id` and cannot be linked to wells.

**Pre-specified direction:** ρ > 0 (higher metal → higher KO density per Mb).  
**Label:** EXPLORATORY throughout.

## Block 0 — Imports and Spark

In [1]:
import sys, re
from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests

PROJECT = Path('/home/hmacgregor/BERIL-research-observatory/projects/comprehensive_metal_ecology')
DATA = PROJECT / 'data'

PRIMARY_METALS = ['Cu', 'Zn', 'Ni']   # BH-FDR applied to these
SECONDARY_METALS = ['Co', 'Cr', 'As']

# Metal keyword patterns for column matching (case-insensitive)
METAL_KEYWORDS = ['copper', 'zinc', 'nickel', 'cobalt', 'chromium', 'arsenic',
                  'manganese', 'iron', 'lead', 'cadmium', r'\bcu\b', r'\bzn\b',
                  r'\bni\b', r'\bco\b', r'\bcr\b', r'\bas\b', r'\bmn\b', r'\bfe\b']
METAL_RE = re.compile('|'.join(METAL_KEYWORDS), re.IGNORECASE)

print('Imports OK')

Imports OK


In [2]:
spark = get_spark_session()
print(f'Spark version: {spark.version}')

Spark version: 4.0.1


## Step 1 — Extract Well Names for 185 MAGs

In [3]:
# MAGs: browser_genome.sample_id IS NOT NULL, strain_id IS NULL
mag_wells = spark.sql('''
    SELECT
        g.id           AS genome_id,
        g.name         AS genome_name,
        g.size         AS genome_size_bp,
        g.genes        AS n_genes,
        s.sample_id    AS well_id
    FROM enigma_genome_depot_enigma.browser_genome g
    JOIN enigma_genome_depot_enigma.browser_sample s ON s.id = g.sample_id
    WHERE g.sample_id IS NOT NULL
      AND (g.strain_id IS NULL OR g.strain_id = 0)
''').toPandas()

print(f'Total MAGs: {len(mag_wells)}')
print(f'Unique wells: {mag_wells["well_id"].nunique()}')
print()
well_counts = mag_wells.groupby('well_id')['genome_id'].count().sort_values(ascending=False)
print('MAGs per well:')
print(well_counts.to_string())

Total MAGs: 185
Unique wells: 21

MAGs per well:
well_id
FW021       24
GW715       22
FW602       18
FW300       16
DP16D       15
FW306_06    11
FW106-10    11
FW215       10
FW106-02     9
GW928        9
FW305        6
FW306_01     6
FW106        5
FW306_05     5
FW306_04     4
FW104        3
FW301        3
FW306_02     3
FW301-02     3
FW306_03     1
GW199        1


In [4]:
# Store sorted well IDs (longest first) for startswith matching
all_well_ids = sorted(mag_wells['well_id'].unique().tolist(), key=len, reverse=True)
print('All MAG well IDs (longest first):')
print(all_well_ids)

def _match_well(sample_name, well_ids):
    for w in well_ids:
        if str(sample_name).startswith(w):
            return w
    return None

All MAG well IDs (longest first):
['FW106-02', 'FW106-10', 'FW301-02', 'FW306_01', 'FW306_02', 'FW306_03', 'FW306_04', 'FW306_05', 'FW306_06', 'DP16D', 'FW021', 'FW104', 'FW106', 'FW215', 'FW300', 'FW301', 'FW305', 'FW602', 'GW199', 'GW715', 'GW928']


## Step 2 — Geochemistry Table Discovery

In [5]:
# 2a. List all tables in enigma_coral
enigma_coral_tables = spark.sql('SHOW TABLES IN enigma_coral').toPandas()
print(f'enigma_coral tables ({len(enigma_coral_tables)}):')
print(enigma_coral_tables.to_string(index=False))

enigma_coral tables (693):
   namespace          tableName  isTemporary
enigma_coral  sys_process_input        False
enigma_coral sys_process_output        False
enigma_coral   ddt_brick0000006        False
enigma_coral   ddt_brick0000007        False
enigma_coral   ddt_brick0000008        False
enigma_coral   ddt_brick0000009        False
enigma_coral   ddt_brick0000010        False
enigma_coral   ddt_brick0000011        False
enigma_coral   ddt_brick0000012        False
enigma_coral   ddt_brick0000013        False
enigma_coral   ddt_brick0000014        False
enigma_coral   ddt_brick0000015        False
enigma_coral   ddt_brick0000016        False
enigma_coral   ddt_brick0000017        False
enigma_coral   ddt_brick0000018        False
enigma_coral   ddt_brick0000019        False
enigma_coral   ddt_brick0000020        False
enigma_coral   ddt_brick0000021        False
enigma_coral   ddt_brick0000022        False
enigma_coral   ddt_brick0000023        False
enigma_coral   ddt_brick0000

In [6]:
# 2b. Describe each enigma_coral table and flag those with metal-like columns
geo_candidate_tables = []  # (namespace, table, metal_cols_found, sample_col)

for _, row in enigma_coral_tables.iterrows():
    tbl = row.get('tableName', row.get('table_name', str(row.iloc[0])))
    if not tbl or str(tbl).startswith('#'):
        continue
    try:
        desc = spark.sql(f'DESCRIBE TABLE enigma_coral.{tbl}').toPandas()
        cols = desc['col_name'].tolist()
        metal_cols = [c for c in cols if METAL_RE.search(str(c))]
        sample_cols = [c for c in cols if any(k in str(c).lower() for k in ['sample', 'well', 'location'])]
        print(f'  {tbl}: {len(cols)} cols, {len(metal_cols)} metal-like, {len(sample_cols)} sample-like')
        if metal_cols:
            print(f'    Metal cols: {metal_cols[:8]}')
            print(f'    Sample cols: {sample_cols[:5]}')
        if metal_cols and sample_cols:
            geo_candidate_tables.append({'ns': 'enigma_coral', 'table': tbl,
                                          'metal_cols': metal_cols, 'sample_cols': sample_cols})
    except Exception as e:
        print(f'  {tbl}: ERROR — {e}')

print(f'\nCandidate geochemistry tables: {len(geo_candidate_tables)}')

  sys_process_input: 10 cols, 0 metal-like, 2 sample-like


  sys_process_output: 12 cols, 0 metal-like, 1 sample-like


  ddt_brick0000006: 3 cols, 0 metal-like, 0 sample-like


  ddt_brick0000007: 71 cols, 18 metal-like, 1 sample-like
    Metal cols: ['concentration_molecule_from_list_iron_2_milligram_per_liter', 'concentration_molecule_from_list_cadmium_atom_milligram_per_liter', 'concentration_molecule_from_list_cobalt_atom_milligram_per_liter', 'concentration_molecule_from_list_chromium_atom_milligram_per_liter', 'concentration_molecule_from_list_copper_atom_milligram_per_liter', 'concentration_molecule_from_list_iron_atom_milligram_per_liter', 'concentration_molecule_from_list_manganese_atom_milligram_per_liter', 'concentration_molecule_from_list_nickel_atom_milligram_per_liter']
    Sample cols: ['sdt_sample_name']


  ddt_brick0000008: 4 cols, 1 metal-like, 1 sample-like
    Metal cols: ['concentration_molecule_from_list_iron_2_milligram_per_liter']
    Sample cols: ['sdt_sample_name']


...

  sys_oterm: 8 cols, 0 metal-like, 0 sample-like


  sys_typedef: 14 cols, 0 metal-like, 0 sample-like
  ddt_brick0001600: 5 cols, 0 metal-like, 0 sample-like


  ddt_ndarray: 15 cols, 0 metal-like, 0 sample-like


  sys_ddt_typedef: 15 cols, 0 metal-like, 0 sample-like


  sys_process: 12 cols, 0 metal-like, 0 sample-like

Candidate geochemistry tables: 11


In [7]:
# 2c. Check kbase.nmdc_arkin.abiotic_features
try:
    nmdc_desc = spark.sql('DESCRIBE TABLE kbase.nmdc_arkin.abiotic_features').toPandas()
    nmdc_cols = nmdc_desc['col_name'].tolist()
    metal_like = [c for c in nmdc_cols if METAL_RE.search(str(c))]
    sample_like = [c for c in nmdc_cols if any(k in str(c).lower() for k in ['sample', 'well', 'id', 'location', 'site'])]
    print(f'kbase.nmdc_arkin.abiotic_features: {len(nmdc_cols)} cols')
    print(f'  Metal-like columns ({len(metal_like)}): {metal_like[:10]}')
    print(f'  Sample/ID columns ({len(sample_like)}): {sample_like[:10]}')
    print(f'  All columns: {nmdc_cols[:30]}')
    
    if metal_like or sample_like:
        # Sample a few rows
        sample_rows = spark.sql('SELECT * FROM kbase.nmdc_arkin.abiotic_features LIMIT 5').toPandas()
        print('\nSample rows:')
        print(sample_rows.to_string())
except Exception as e:
    print(f'kbase.nmdc_arkin.abiotic_features: NOT ACCESSIBLE — {e}')

kbase.nmdc_arkin.abiotic_features: 22 cols
  Metal-like columns (1): ['annotations_manganese_has_numeric_value']
  Sample/ID columns (1): ['sample_id']
  All columns: ['sample_id', 'annotations_ammonium_has_numeric_value', 'annotations_ammonium_nitrogen_has_numeric_value', 'annotations_calcium_has_numeric_value', 'annotations_carb_nitro_ratio_has_numeric_value', 'annotations_chlorophyll_has_numeric_value', 'annotations_conduc_has_numeric_value', 'annotations_depth_has_maximum_numeric_value', 'annotations_depth_has_minimum_numeric_value', 'annotations_depth_has_numeric_value', 'annotations_diss_org_carb_has_numeric_value', 'annotations_diss_oxygen_has_numeric_value', 'annotations_magnesium_has_numeric_value', 'annotations_manganese_has_numeric_value', 'annotations_ph', 'annotations_potassium_has_numeric_value', 'annotations_samp_size_has_numeric_value', 'annotations_soluble_react_phosp_has_numeric_value', 'annotations_temp_has_numeric_value', 'annotations_tot_nitro_content_has_numeric_v


Sample rows:
              sample_id  annotations_ammonium_has_numeric_value  annotations_ammonium_nitrogen_has_numeric_value  annotations_calcium_has_numeric_value  annotations_carb_nitro_ratio_has_numeric_value  annotations_chlorophyll_has_numeric_value  annotations_conduc_has_numeric_value  annotations_depth_has_maximum_numeric_value  annotations_depth_has_minimum_numeric_value  annotations_depth_has_numeric_value  annotations_diss_org_carb_has_numeric_value  annotations_diss_oxygen_has_numeric_value  annotations_magnesium_has_numeric_value  annotations_manganese_has_numeric_value  annotations_ph  annotations_potassium_has_numeric_value  annotations_samp_size_has_numeric_value  annotations_soluble_react_phosp_has_numeric_value  annotations_temp_has_numeric_value  annotations_tot_nitro_content_has_numeric_value  annotations_tot_org_carb_has_numeric_value  annotations_tot_phosp_has_numeric_value
0  nmdc:bsm-11-042nd237                                     0.0                          

In [8]:
# 2d. Check nmdc_arkin for FRC-relevant sample IDs
try:
    # Look for any column that might contain FRC well IDs (FW, GW, etc.)
    # Check text columns that might hold sample IDs
    id_cols = [c for c in nmdc_cols if any(k in str(c).lower() for k in ['id', 'name', 'label', 'sample'])]
    print(f'ID-like columns in nmdc_arkin.abiotic_features: {id_cols[:10]}')
    
    if id_cols:
        sample_ids = spark.sql(f'SELECT DISTINCT `{id_cols[0]}` FROM kbase.nmdc_arkin.abiotic_features LIMIT 20').toPandas()
        print(f'Sample {id_cols[0]} values:')
        print(sample_ids.to_string())
        
        # Check whether any FRC well pattern appears
        frc_patterns = ['FW', 'GW', 'FRC', 'oak ridge', 'ORNL']
        id_vals = sample_ids.iloc[:, 0].astype(str).str.upper().tolist()
        frc_hits = [v for v in id_vals if any(p in v for p in ['FW', 'GW', 'FRC'])]
        print(f'\nFRC-pattern hits: {frc_hits[:10]}')
except Exception as e:
    print(f'nmdc_arkin ID check: {e}')

ID-like columns in nmdc_arkin.abiotic_features: ['sample_id']


Sample sample_id values:
               sample_id
0   nmdc:bsm-11-7qaw2x10
1   nmdc:bsm-11-8qdykb22
2   nmdc:bsm-11-cmesvc96
3   nmdc:bsm-11-dcrnt111
4   nmdc:bsm-11-qbd0vv07
5   nmdc:bsm-11-hdghyn80
6   nmdc:bsm-11-0b3f6q91
7   nmdc:bsm-11-2769z821
8   nmdc:bsm-11-3qhtv606
9   nmdc:bsm-11-4p9p1529
10  nmdc:bsm-11-4qsxse29
11  nmdc:bsm-11-5x1h9m56
12  nmdc:bsm-11-6a50sg07
13  nmdc:bsm-11-6s3yav56
14  nmdc:bsm-11-9qzbhv51
15  nmdc:bsm-11-9wzd6677
16  nmdc:bsm-11-axm9cz50
17  nmdc:bsm-11-b9thcs44
18  nmdc:bsm-11-bzpdcd19
19  nmdc:bsm-11-d03x1f58

FRC-pattern hits: []


In [9]:
# 2e. Profile ddt_brick0000007 for comparison (the NB11 source)
# Count how many unique sample names it has, and how many match MAG wells
try:
    ddt_samples = spark.sql('''
        SELECT DISTINCT sdt_sample_name
        FROM enigma_coral.ddt_brick0000007
        WHERE sdt_sample_name IS NOT NULL
    ''').toPandas()
    ddt_samples['matched_well'] = ddt_samples['sdt_sample_name'].apply(
        lambda x: _match_well(str(x), all_well_ids)
    )
    n_matched = ddt_samples['matched_well'].notna().sum()
    wells_matched = ddt_samples.dropna(subset=['matched_well'])['matched_well'].unique().tolist()
    print(f'ddt_brick0000007: {len(ddt_samples)} unique sample names')
    print(f'Matching MAG wells: {n_matched} rows → {len(wells_matched)} wells')
    print(f'Wells: {sorted(wells_matched)}')
except Exception as e:
    print(f'ddt_brick0000007 profile: {e}')

ddt_brick0000007: 300 unique sample names
Matching MAG wells: 121 rows → 10 wells
Wells: ['DP16D', 'FW021', 'FW104', 'FW106', 'FW106-02', 'FW215', 'FW300', 'FW301', 'FW305', 'FW602']


In [10]:
# 2f. Profile other candidate enigma_coral tables for MAG well overlap
table_coverage = []  # (table, n_wells_matched, total_rows)

# Known sample column candidates per table (will be detected dynamically)
for cand in geo_candidate_tables:
    ns, tbl = cand['ns'], cand['table']
    if tbl == 'ddt_brick0000007':
        # Already profiled
        table_coverage.append({'table': tbl, 'n_wells_matched': len(wells_matched),
                                'n_mag_rows': n_matched, 'sample_col': 'sdt_sample_name'})
        continue
    
    for scol in cand['sample_cols'][:3]:  # try first 3 sample-like cols
        try:
            sample_vals = spark.sql(f'''
                SELECT DISTINCT `{scol}` FROM {ns}.{tbl}
                WHERE `{scol}` IS NOT NULL
                LIMIT 1000
            ''').toPandas()
            matched = sample_vals[scol].apply(
                lambda x: _match_well(str(x), all_well_ids)
            ).notna().sum()
            matched_wells = sample_vals[scol].apply(
                lambda x: _match_well(str(x), all_well_ids)
            ).dropna().unique().tolist()
            print(f'  {tbl} [{scol}]: {matched} matching sample-names → {len(matched_wells)} wells: {sorted(matched_wells)}')
            if matched > 0:
                table_coverage.append({'table': f'{ns}.{tbl}', 'n_wells_matched': len(matched_wells),
                                        'n_mag_rows': matched, 'sample_col': scol,
                                        'metal_cols': cand['metal_cols']})
                break
        except Exception as e:
            print(f'  {tbl} [{scol}]: {e}')

print(f'\nTable coverage summary:')
for t in sorted(table_coverage, key=lambda x: -x['n_wells_matched']):
    print(f"  {t['table']}: {t['n_wells_matched']} wells, {t['n_mag_rows']} matching rows")

  ddt_brick0000008 [sdt_sample_name]: 18 matching sample-names → 11 wells: ['DP16D', 'FW021', 'FW104', 'FW106', 'FW215', 'FW300', 'FW301', 'FW602', 'GW199', 'GW715', 'GW928']


  ddt_brick0000052 [sdt_sample_name]: 0 matching sample-names → 0 wells: []


  ddt_brick0000052 [environmental_sample_weight_category_subsample_gram]: 0 matching sample-names → 0 wells: []


  ddt_brick0000052 [environmental_sample_volume_category_diluted_sample_volume_milliliter]: 0 matching sample-names → 0 wells: []


  ddt_brick0000053 [sdt_sample_name]: 1 matching sample-names → 1 wells: ['FW106']


...

  ddt_brick0000068 [environmental_sample_control_name]: 0 matching sample-names → 0 wells: []


  ddt_brick0000069 [sdt_sample_name]: 0 matching sample-names → 0 wells: []


  ddt_brick0000069 [environmental_sample_depth_meter]: 0 matching sample-names → 0 wells: []


  ddt_brick0000069 [volume_category_water_sample_volume_liter]: 0 matching sample-names → 0 wells: []


  ddt_brick0000081 [sdt_sample_name]: 24 matching sample-names → 11 wells: ['DP16D', 'FW021', 'FW104', 'FW106', 'FW215', 'FW300', 'FW301', 'FW602', 'GW199', 'GW715', 'GW928']

Table coverage summary:
  enigma_coral.ddt_brick0000008: 11 wells, 18 matching rows
  enigma_coral.ddt_brick0000081: 11 wells, 24 matching rows
  ddt_brick0000007: 10 wells, 121 matching rows
  enigma_coral.ddt_brick0000054: 2 wells, 2 matching rows
  enigma_coral.ddt_brick0000057: 2 wells, 2 matching rows
  enigma_coral.ddt_brick0000053: 1 wells, 1 matching rows
  enigma_coral.ddt_brick0000056: 1 wells, 1 matching rows


In [11]:
# Fix: ddt_brick0000007 is the ONLY enigma_coral table with Cu/Zn/Ni.
# ddt_brick0000008 covers 11 wells but has only Fe2+ — a naive "most wells"
# criterion selects it, leaving metals_to_test empty and crashing the correlations.
# Always use ddt_brick0000007 for primary metal analysis.
# ddt_brick0000008 is used separately below for a supplementary Fe2+ analysis.

best_table = 'enigma_coral.ddt_brick0000007'
best_sample_col = 'sdt_sample_name'
best_n_wells = len(wells_matched)  # 10 wells from nb140013

# Supplementary Fe2+ analysis
SUPPLEMENTARY_FE_TABLE = 'enigma_coral.ddt_brick0000008'
SUPPLEMENTARY_FE_COL = 'sdt_sample_name'
FE_METAL_COL = 'concentration_molecule_from_list_iron_2_milligram_per_liter'

print(f'PRIMARY ANALYSIS: {best_table}')
print(f'  Wells: {best_n_wells} | Metals: Cu, Zn, Ni, Co, Cr, As, Mn')
print()
print(f'SUPPLEMENTARY: {SUPPLEMENTARY_FE_TABLE}')
print(f'  Wells: 11 | Metal: Fe2+ only')
print(f'  (excluded from primary — lacks Cu/Zn/Ni; analysed separately below)')
print(f'\nFinal primary selection: {best_table}')

PRIMARY ANALYSIS: enigma_coral.ddt_brick0000007
  Wells: 10 | Metals: Cu, Zn, Ni, Co, Cr, As, Mn

SUPPLEMENTARY: enigma_coral.ddt_brick0000008
  Wells: 11 | Metal: Fe2+ only
  (excluded from primary — lacks Cu/Zn/Ni; analysed separately below)

Final primary selection: enigma_coral.ddt_brick0000007


## Step 3 — Fetch Geochemistry from Selected Source

In [12]:
# Pre-specified metal column mapping for ddt_brick0000007
DDT_METAL_COLS = {
    'Cu': 'concentration_molecule_from_list_copper_atom_milligram_per_liter',
    'Ni': 'concentration_molecule_from_list_nickel_atom_milligram_per_liter',
    'Zn': 'concentration_molecule_from_list_zinc_atom_milligram_per_liter',
    'As': 'concentration_molecule_from_list_arsane_milligram_per_liter',
    'Mn': 'concentration_molecule_from_list_manganese_atom_milligram_per_liter',
    'Cr': 'concentration_molecule_from_list_chromium_atom_milligram_per_liter',
    'Co': 'concentration_molecule_from_list_cobalt_atom_milligram_per_liter',
}

# If a different table was selected, its metal columns need to be identified dynamically.
# For ddt_brick0000007, use the known mapping.
if best_table == 'enigma_coral.ddt_brick0000007':
    ACTIVE_METAL_COLS = DDT_METAL_COLS
    metal_select = ', '.join(f'{v} AS {k}' for k, v in ACTIVE_METAL_COLS.items())
    geo_raw = spark.sql(f'''
        SELECT {best_sample_col} AS sample_name, {metal_select}
        FROM {best_table}
        WHERE {best_sample_col} IS NOT NULL
    ''').toPandas()
else:
    # For a discovered table: pull all metal-like columns dynamically
    discovered_metal_cols = [c for c in best.get('metal_cols', []) if c]
    if not discovered_metal_cols:
        raise ValueError(f'No metal columns identified in {best_table}')
    metal_select = ', '.join(f'`{c}`' for c in discovered_metal_cols)
    ACTIVE_METAL_COLS = {c: c for c in discovered_metal_cols}  # use col names as keys
    geo_raw = spark.sql(f'''
        SELECT `{best_sample_col}` AS sample_name, {metal_select}
        FROM {best_table}
        WHERE `{best_sample_col}` IS NOT NULL
    ''').toPandas()

print(f'Geochemistry rows fetched: {len(geo_raw)}')
print(f'Metal columns: {list(ACTIVE_METAL_COLS.keys())}')
print(geo_raw.head(3).to_string())

Geochemistry rows fetched: 300
Metal columns: ['Cu', 'Ni', 'Zn', 'As', 'Mn', 'Cr', 'Co']
        sample_name        Cu        Ni        Zn  As        Mn  Cr  Co
0    GW460-11-04-13  0.010386  0.000127  0.050223 NaN  0.005236 NaN NaN
1    GW456-11-04-13       NaN       NaN       NaN NaN       NaN NaN NaN
2  GW456-11-04-13-2  0.000831  0.003616  0.037924 NaN  0.011673 NaN NaN


In [13]:
# Match sample names to MAG well IDs
geo_raw['well_id'] = geo_raw['sample_name'].apply(
    lambda x: _match_well(str(x), all_well_ids)
)
matched_rows = geo_raw['well_id'].notna().sum()
matched_wells_final = sorted(geo_raw.dropna(subset=['well_id'])['well_id'].unique().tolist())
print(f'Geochemistry rows matched to MAG wells: {matched_rows}/{len(geo_raw)} ({100*matched_rows/len(geo_raw):.1f}%)')
print(f'Unique wells matched: {len(matched_wells_final)}')
print(f'Wells: {matched_wells_final}')

# How many MAGs are in the matched wells?
mags_in_matched_wells = mag_wells[mag_wells['well_id'].isin(matched_wells_final)]
print(f'\nMAGs in matched wells: {len(mags_in_matched_wells)}/{len(mag_wells)}')
print('Per-well MAG count:')
print(mags_in_matched_wells.groupby('well_id')['genome_id'].count().sort_values(ascending=False).to_string())

Geochemistry rows matched to MAG wells: 121/300 (40.3%)
Unique wells matched: 10
Wells: ['DP16D', 'FW021', 'FW104', 'FW106', 'FW106-02', 'FW215', 'FW300', 'FW301', 'FW305', 'FW602']

MAGs in matched wells: 109/185
Per-well MAG count:
well_id
FW021       24
FW602       18
FW300       16
DP16D       15
FW215       10
FW106-02     9
FW305        6
FW106        5
FW104        3
FW301        3


In [14]:
# Well-level median across time points
active_metals = list(ACTIVE_METAL_COLS.keys())

geo_well = (
    geo_raw
    .dropna(subset=['well_id'])
    .groupby('well_id')[active_metals]
    .median()
    .reset_index()
)

geo_well_counts = (
    geo_raw
    .dropna(subset=['well_id'])
    .groupby('well_id')[active_metals]
    .count()
    .reset_index()
    .rename(columns={m: f'{m}_n' for m in active_metals})
)
geo_well = geo_well.merge(geo_well_counts, on='well_id')

print(f'Wells with geochemistry data: {len(geo_well)}')
print(geo_well[['well_id'] + active_metals[:5]].to_string(index=False))

Wells with geochemistry data: 10
 well_id       Cu       Ni       Zn        As       Mn
   DP16D      NaN      NaN      NaN       NaN      NaN
   FW021      NaN      NaN      NaN       NaN      NaN
   FW104      NaN      NaN      NaN       NaN      NaN
   FW106      NaN      NaN      NaN       NaN      NaN
FW106-02      NaN      NaN      NaN       NaN      NaN
   FW215 0.001400 0.002700 0.002200 14.170000 0.120000
   FW300 0.002757 0.002817 0.047056  0.002035 0.017214
   FW301 0.003536 0.005213 0.037826  0.001541 0.014519
   FW305      NaN      NaN      NaN       NaN      NaN
   FW602      NaN      NaN      NaN       NaN      NaN


## Step 4 — Compute per-Mb 140-KO Density for MAGs

In [15]:
# Load 140 primary KOs
# Fix: str.startswith(('Tier 1', 'Tier 2')) also matches 'Tier 2-Fitness' (116 KOs)
# → 256 total. Use exact membership instead.
gl = pd.read_csv(DATA / 'curated_mrg_ko_ids_v2.csv')
primary_kos = set(
    gl.loc[gl['evidence_tier'].isin(['Tier 1', 'Tier 2']), 'KO'].str.strip()
)
print(f'Primary KO set: {len(primary_kos)} KOs')
print(f'Expected: 140 (Tier 1: 32, Tier 2: 108)')
print(f'Tier counts:\n{gl.groupby("evidence_tier")["KO"].count().to_string()}')

Primary KO set: 140 KOs
Expected: 140 (Tier 1: 32, Tier 2: 108)
Tier counts:
evidence_tier
Tier 1             32
Tier 2            108
Tier 2-Fitness    116
Tier 3            286
Tier 3-BacMet     188


In [16]:
# Detect KO format
r_ko_fmt = spark.sql('''
    SELECT kegg_id FROM enigma_genome_depot_enigma.browser_kegg_ortholog LIMIT 3
''').toPandas()
sample_kegg_id = r_ko_fmt['kegg_id'].iloc[0]
if sample_kegg_id.startswith('ko:'):
    ko_sql_vals = ','.join(f"'ko:{k}'" for k in primary_kos)
else:
    ko_sql_vals = ','.join(f"'{k}'" for k in primary_kos)
print(f'KO format: {repr(sample_kegg_id)} → using bare format: {"ko:" not in ko_sql_vals[:10]}')

KO format: 'K02313' → using bare format: True


In [17]:
# Query: distinct primary KOs per MAG
mag_ko_query = f'''
    SELECT
        g.id          AS genome_id,
        g.size        AS genome_size_bp,
        s.sample_id   AS well_id,
        COUNT(DISTINCT ko.kegg_id) AS n_primary_ko
    FROM enigma_genome_depot_enigma.browser_genome g
    JOIN enigma_genome_depot_enigma.browser_sample s
        ON s.id = g.sample_id
    JOIN enigma_genome_depot_enigma.browser_gene gene
        ON gene.genome_id = g.id
    JOIN enigma_genome_depot_enigma.browser_protein_kegg_orthologs pko
        ON pko.protein_id = gene.protein_id
    JOIN enigma_genome_depot_enigma.browser_kegg_ortholog ko
        ON ko.id = pko.kegg_ortholog_id
    WHERE
        g.sample_id IS NOT NULL
        AND (g.strain_id IS NULL OR g.strain_id = 0)
        AND ko.kegg_id IN ({ko_sql_vals})
    GROUP BY g.id, g.size, s.sample_id
'''

mag_ko_df = spark.sql(mag_ko_query).toPandas()
print(f'MAGs with ≥1 primary KO: {len(mag_ko_df)}')
print(mag_ko_df.head())

MAGs with ≥1 primary KO: 184
   genome_id  genome_size_bp   well_id  n_primary_ko
0        496         3526527     FW602            20
1         85         2970669     FW021            24
2        137         3379963  FW106-10            26
3       3761         3399089  FW306_02            19
4        133         3235322  FW106-10            23


In [18]:
# Left-join to all MAGs (include 0-KO MAGs)
mag_density = mag_wells.merge(
    mag_ko_df[['genome_id', 'n_primary_ko']], on='genome_id', how='left'
)
mag_density['n_primary_ko'] = mag_density['n_primary_ko'].fillna(0).astype(int)
mag_density['genome_size_mb'] = mag_density['genome_size_bp'] / 1e6
mag_density['ko_per_mb'] = mag_density.apply(
    lambda r: r['n_primary_ko'] / r['genome_size_mb'] if r['genome_size_mb'] > 0 else np.nan,
    axis=1
)

print(f'All MAGs: {len(mag_density)}')
print(f'KO density summary:')
print(mag_density['ko_per_mb'].describe().round(3))

All MAGs: 185
KO density summary:
count    185.000
mean       5.444
std        2.086
min        0.000
25%        4.081
50%        5.371
75%        6.672
max       12.086
Name: ko_per_mb, dtype: float64


## Step 5 — Join MAGs to Geochemistry

In [19]:
# MAG-level join
mag_geo = mag_density.merge(geo_well, on='well_id', how='inner')

print(f'MAGs with geochemistry: {len(mag_geo)}')
print(f'Unique wells in joined dataset: {mag_geo["well_id"].nunique()}')
print(f'MAGs dropped (no geo match): {len(mag_density) - len(mag_geo)}')
print()

well_summary = (
    mag_geo.groupby('well_id')
    .agg(n_mags=('genome_id', 'count'),
         mean_ko_per_mb=('ko_per_mb', 'mean'),
         median_ko_per_mb=('ko_per_mb', 'median'))
    .reset_index()
)
print('Per-well summary:')
print(well_summary.to_string(index=False))

MAGs with geochemistry: 109
Unique wells in joined dataset: 10
MAGs dropped (no geo match): 76

Per-well summary:
 well_id  n_mags  mean_ko_per_mb  median_ko_per_mb
   DP16D      15        5.978536          6.590448
   FW021      24        5.864887          5.863595
   FW104       3        8.119610          8.195756
   FW106       5        5.956929          5.980206
FW106-02       9        6.964648          6.918361
   FW215      10        3.076249          3.444237
   FW300      16        4.800111          5.048254
   FW301       3        4.428804          4.753386
   FW305       6        5.325475          4.778743
   FW602      18        5.301274          5.521013


In [20]:
# Feasibility check
n_mags_geo = len(mag_geo)
n_wells_geo = mag_geo['well_id'].nunique()
n_wells_ge2 = (mag_geo.groupby('well_id')['genome_id'].count() >= 2).sum()

print(f'Feasibility:')
print(f'  MAGs with geo data: {n_mags_geo}')
print(f'  Wells: {n_wells_geo}')
print(f'  Wells with ≥2 MAGs: {n_wells_ge2}')

if n_mags_geo < 5:
    print('INSUFFICIENT DATA: fewer than 5 MAGs matched. Results reported as underpowered.')
elif n_wells_geo < 5:
    print(f'WARNING: only {n_wells_geo} wells with data. Well-level correlations will be underpowered.')

Feasibility:
  MAGs with geo data: 109
  Wells: 10
  Wells with ≥2 MAGs: 10


## Step 6 — Correlation Analysis

In [21]:
def spearman_row(data, metal, density_col='ko_per_mb', level='mag'):
    if metal not in data.columns:
        return {'metal': metal, 'level': level, 'n': 0, 'rho': np.nan,
                'p_raw': np.nan, 'notes': 'column not in dataset'}
    sub = data[[density_col, metal]].dropna()
    n = len(sub)
    if n < 5:
        return {'metal': metal, 'level': level, 'n': n, 'rho': np.nan,
                'p_raw': np.nan, 'notes': f'n<5'}
    rho, p = stats.spearmanr(sub[density_col], sub[metal])
    return {'metal': metal, 'level': level, 'n': n,
            'rho': round(float(rho), 4), 'p_raw': round(float(p), 4), 'notes': ''}

# Test all available metals (primary first, then secondary)
metals_to_test = [m for m in PRIMARY_METALS + SECONDARY_METALS + ['Mn']
                  if m in active_metals]
print(f'Metals to test: {metals_to_test}')

# MAG-level
mag_results = [spearman_row(mag_geo, m, density_col='ko_per_mb', level='mag')
               for m in metals_to_test]
mag_corr = pd.DataFrame(mag_results)

# BH-FDR on primary metals only
primary_mask = mag_corr['metal'].isin(PRIMARY_METALS) & mag_corr['p_raw'].notna()
if primary_mask.sum() > 0:
    _, q_bh, _, _ = multipletests(mag_corr.loc[primary_mask, 'p_raw'].values, method='fdr_bh')
    mag_corr.loc[primary_mask, 'p_fdr'] = q_bh.round(4)

print('MAG-level Spearman:')
print(mag_corr.to_string(index=False))

Metals to test: ['Cu', 'Zn', 'Ni', 'Co', 'Cr', 'As', 'Mn']
MAG-level Spearman:
metal level  n     rho  p_raw notes  p_fdr
   Cu   mag 29  0.3024 0.1108       0.1108
   Zn   mag 29  0.3722 0.0468       0.1108
   Ni   mag 29  0.3024 0.1108       0.1108
   Co   mag 19 -0.1318 0.5908          NaN
   Cr   mag 29 -0.3024 0.1108          NaN
   As   mag 29 -0.3024 0.1108          NaN
   Mn   mag 29 -0.3024 0.1108          NaN


In [22]:
# Well-level (≥2 MAGs per well)
well_density = (
    mag_geo.groupby('well_id')
    .agg(n_mags=('genome_id', 'count'),
         median_ko_per_mb=('ko_per_mb', 'median'))
    .reset_index()
)
well_geo = well_density[well_density['n_mags'] >= 2].merge(geo_well, on='well_id', how='inner')

print(f'Well-level dataset (n_mags≥2): {len(well_geo)} wells')
if len(well_geo) > 0:
    print(well_geo[['well_id', 'n_mags', 'median_ko_per_mb'] + metals_to_test[:4]].to_string(index=False))

well_results = [spearman_row(well_geo, m, density_col='median_ko_per_mb', level='well')
                for m in metals_to_test]
well_corr = pd.DataFrame(well_results)

primary_mask_w = well_corr['metal'].isin(PRIMARY_METALS) & well_corr['p_raw'].notna()
if primary_mask_w.sum() > 0:
    _, q_bh_w, _, _ = multipletests(well_corr.loc[primary_mask_w, 'p_raw'].values, method='fdr_bh')
    well_corr.loc[primary_mask_w, 'p_fdr'] = q_bh_w.round(4)

print('\nWell-level Spearman:')
print(well_corr.to_string(index=False))

Well-level dataset (n_mags≥2): 10 wells
 well_id  n_mags  median_ko_per_mb       Cu       Zn       Ni       Co
   DP16D      15          6.590448      NaN      NaN      NaN      NaN
   FW021      24          5.863595      NaN      NaN      NaN      NaN
   FW104       3          8.195756      NaN      NaN      NaN      NaN
   FW106       5          5.980206      NaN      NaN      NaN      NaN
FW106-02       9          6.918361      NaN      NaN      NaN      NaN
   FW215      10          3.444237 0.001400 0.002200 0.002700      NaN
   FW300      16          5.048254 0.002757 0.047056 0.002817 0.001901
   FW301       3          4.753386 0.003536 0.037826 0.005213 0.003145
   FW305       6          4.778743      NaN      NaN      NaN      NaN
   FW602      18          5.521013      NaN      NaN      NaN      NaN

Well-level Spearman:
metal level  n  rho  p_raw notes
   Cu  well  3  NaN    NaN   n<5
   Zn  well  3  NaN    NaN   n<5
   Ni  well  3  NaN    NaN   n<5
   Co  well  2  NaN    Na

In [23]:
# Combined metal burden (mean z-scored well-level concentrations)
burden_metals = [m for m in metals_to_test if m in well_geo.columns
                 and well_geo[m].notna().sum() >= 3]

if burden_metals and len(well_geo) >= 3:
    burden_df = well_geo[burden_metals].copy()
    # Z-score each metal
    for m in burden_metals:
        mu, sd = burden_df[m].mean(), burden_df[m].std()
        burden_df[f'{m}_z'] = (burden_df[m] - mu) / sd if sd > 0 else 0
    well_geo['combined_burden'] = burden_df[[f'{m}_z' for m in burden_metals]].mean(axis=1)

    burden_row = spearman_row(well_geo, 'combined_burden', density_col='median_ko_per_mb', level='well-burden')
    print(f'Combined burden ({len(burden_metals)} metals): ρ={burden_row["rho"]}, p={burden_row["p_raw"]}, n={burden_row["n"]}')
else:
    burden_row = {'metal': 'combined_burden', 'level': 'well-burden', 'n': len(well_geo),
                  'rho': np.nan, 'p_raw': np.nan, 'notes': 'insufficient wells for burden'}
    print('Combined burden: insufficient data.')

Combined burden (6 metals): ρ=nan, p=nan, n=3


## Step 6b — Supplementary Fe2+ Analysis (ddt_brick0000008, 11 wells)

`ddt_brick0000008` covers 11 wells (vs 10 for the primary source) but holds only Fe2+.
Ferrous iron reflects reducing conditions. Included as supplementary context only — Fe is
not a primary metal for this hypothesis.

In [24]:
# Fetch Fe2+ from ddt_brick0000008 (supplementary; 11 wells)
fe_raw = spark.sql(f'''
    SELECT {SUPPLEMENTARY_FE_COL} AS sample_name,
           `{FE_METAL_COL}` AS Fe
    FROM {SUPPLEMENTARY_FE_TABLE}
    WHERE {SUPPLEMENTARY_FE_COL} IS NOT NULL
''').toPandas()

fe_raw['well_id'] = fe_raw['sample_name'].apply(
    lambda x: _match_well(str(x), all_well_ids)
)
fe_well_med = (
    fe_raw.dropna(subset=['well_id'])
    .groupby('well_id')[['Fe']]
    .median()
    .reset_index()
)

print(f'Fe2+ supplementary: {len(fe_raw)} rows → {len(fe_well_med)} wells')
print(fe_well_med.to_string(index=False))

Fe2+ supplementary: 115 rows → 11 wells
well_id    Fe
  DP16D 1.460
  FW021 0.055
  FW104 1.445
  FW106 1.210
  FW215 0.210
  FW300 1.260
  FW301 0.050
  FW602 0.000
  GW199 0.070
  GW715 0.010
  GW928 0.840


In [25]:
# Spearman ρ for Fe2+ vs 140-KO density
fe_mag_geo = mag_density.merge(fe_well_med, on='well_id', how='inner')
fe_n_mags = len(fe_mag_geo)
fe_n_wells = fe_mag_geo['well_id'].nunique()

fe_results = []
fe_results.append(spearman_row(fe_mag_geo, 'Fe', density_col='ko_per_mb', level='mag-supp'))

fe_well_agg = (
    fe_mag_geo.groupby('well_id')
    .agg(n_mags=('genome_id', 'count'),
         median_ko_per_mb=('ko_per_mb', 'median'),
         Fe=('Fe', 'median'))
    .reset_index()
)
fe_results.append(spearman_row(
    fe_well_agg[fe_well_agg['n_mags'] >= 2],
    'Fe', density_col='median_ko_per_mb', level='well-supp'
))

fe_df = pd.DataFrame(fe_results)
for r in fe_results:
    r['source_table'] = SUPPLEMENTARY_FE_TABLE
    r['n_wells'] = fe_n_wells
    r['p_fdr'] = np.nan

print(f'Fe2+ supplementary: {fe_n_mags} MAGs, {fe_n_wells} wells')
print(fe_df[['metal', 'level', 'n', 'rho', 'p_raw']].to_string(index=False))

Fe2+ supplementary: 126 MAGs, 11 wells
metal     level   n    rho  p_raw
   Fe  mag-supp 126 0.1081 0.2284
   Fe well-supp  10 0.5152 0.1276


## Step 7 — Save and Report

In [26]:
# Combine all results (primary metals + Fe2+ supplementary)
all_results = []
for row in mag_results + well_results:
    r = dict(row)
    r['source_table'] = best_table
    r['n_wells'] = n_wells_geo
    if r['level'] == 'mag':
        match = mag_corr[(mag_corr['metal'] == r['metal']) & (mag_corr['level'] == 'mag')]
    else:
        match = well_corr[(well_corr['metal'] == r['metal']) & (well_corr['level'] == 'well')]
    r['p_fdr'] = match['p_fdr'].values[0] if 'p_fdr' in match.columns and len(match) > 0 else np.nan
    all_results.append(r)

all_results.append({**burden_row, 'source_table': best_table, 'n_wells': n_wells_geo, 'p_fdr': np.nan})

# Append supplementary Fe2+ results
all_results.extend(fe_results)

out_df = pd.DataFrame(all_results)
out_path = DATA / 'enigma_geochem_discovery_results.csv'
out_df.to_csv(out_path, index=False)
print(f'Saved to {out_path}')
print(out_df[['metal', 'level', 'n', 'rho', 'p_raw', 'p_fdr', 'source_table']].to_string(index=False))

Saved to /home/hmacgregor/BERIL-research-observatory/projects/comprehensive_metal_ecology/data/enigma_geochem_discovery_results.csv
          metal       level   n     rho  p_raw  p_fdr                  source_table
             Cu         mag  29  0.3024 0.1108 0.1108 enigma_coral.ddt_brick0000007
             Zn         mag  29  0.3722 0.0468 0.1108 enigma_coral.ddt_brick0000007
             Ni         mag  29  0.3024 0.1108 0.1108 enigma_coral.ddt_brick0000007
             Co         mag  19 -0.1318 0.5908    NaN enigma_coral.ddt_brick0000007
             Cr         mag  29 -0.3024 0.1108    NaN enigma_coral.ddt_brick0000007
             As         mag  29 -0.3024 0.1108    NaN enigma_coral.ddt_brick0000007
             Mn         mag  29 -0.3024 0.1108    NaN enigma_coral.ddt_brick0000007
             Cu        well   3     NaN    NaN    NaN enigma_coral.ddt_brick0000007
             Zn        well   3     NaN    NaN    NaN enigma_coral.ddt_brick0000007
             Ni        well 

In [27]:
# Summary report
print('=' * 70)
print('ENIGMA MAG GEOCHEMISTRY DISCOVERY — SUMMARY REPORT')
print('=' * 70)
print()
print(f'Geochemistry tables found in enigma_coral: {len(enigma_coral_tables)}')
print(f'Tables with metal AND sample columns: {len(geo_candidate_tables)}')
print(f'Selected primary source: {best_table}')
print(f'  Reason: only enigma_coral table with primary metals (Cu/Zn/Ni), {best_n_wells} wells')
print(f'  Note: ddt_brick0000008 had 11 wells but only Fe2+ — used as supplementary')
print()
print(f'PRIMARY ANALYSIS — Wells: {n_wells_geo}, MAGs: {n_mags_geo}/{len(mag_wells)}')
print()
print('Comparison with NB11 (ddt_brick0000007, 3 wells, 29 MAGs):')
print(f'  NB11 MAG-level Zn: ρ=+0.380 (p_raw=0.042, FDR NS)')
print(f'  NB11 Cr: ρ=−0.407, combined burden: ρ=−0.407 (p=0.028)')
print()
print('NB14 primary results (MAG-level):')
for _, row in mag_corr[mag_corr['metal'].isin(PRIMARY_METALS)].iterrows():
    dir_ok = 'consistent' if pd.notna(row['rho']) and row['rho'] > 0 else (
        'inconsistent' if pd.notna(row['rho']) else 'NA')
    print(f"  {row['metal']}: ρ={row['rho']}, p={row['p_raw']}, n={row['n']} [{dir_ok}]")
print()
print('NB14 supplementary Fe2+ results:')
for _, row in fe_df.iterrows():
    print(f"  Fe ({row['level']}): ρ={row['rho']}, p={row['p_raw']}, n={row['n']}")

ENIGMA MAG GEOCHEMISTRY DISCOVERY — SUMMARY REPORT

Geochemistry tables found in enigma_coral: 693
Tables with metal AND sample columns: 11
Selected primary source: enigma_coral.ddt_brick0000007
  Reason: only enigma_coral table with primary metals (Cu/Zn/Ni), 10 wells
  Note: ddt_brick0000008 had 11 wells but only Fe2+ — used as supplementary

PRIMARY ANALYSIS — Wells: 10, MAGs: 109/185

Comparison with NB11 (ddt_brick0000007, 3 wells, 29 MAGs):
  NB11 MAG-level Zn: ρ=+0.380 (p_raw=0.042, FDR NS)
  NB11 Cr: ρ=−0.407, combined burden: ρ=−0.407 (p=0.028)

NB14 primary results (MAG-level):
  Cu: ρ=0.3024, p=0.1108, n=29 [consistent]
  Zn: ρ=0.3722, p=0.0468, n=29 [consistent]
  Ni: ρ=0.3024, p=0.1108, n=29 [consistent]

NB14 supplementary Fe2+ results:
  Fe (mag-supp): ρ=0.1081, p=0.2284, n=126
  Fe (well-supp): ρ=0.5152, p=0.1276, n=10


In [28]:
# Update INTERPRETATION_TABLE.md
interp_path = PROJECT / 'INTERPRETATION_TABLE.md'
with open(interp_path, 'r') as f:
    content = f.read()

# Build table rows (primary metals, MAG level)
table_rows = []
for _, row in pd.concat([mag_corr, well_corr]).iterrows():
    if not row['metal'] in PRIMARY_METALS + SECONDARY_METALS:
        continue
    rho = f"{row['rho']:.4f}" if pd.notna(row.get('rho')) else 'NA'
    p = f"{row['p_raw']:.4g}" if pd.notna(row.get('p_raw')) else 'NA'
    q = f"{row.get('p_fdr', np.nan):.4g}" if pd.notna(row.get('p_fdr', np.nan)) else 'NA'
    n = int(row['n'])
    dir_ok = 'Yes' if pd.notna(row.get('rho')) and row['rho'] > 0 else ('No' if pd.notna(row.get('rho')) else 'NA')
    table_rows.append(f"| {row['metal']} | {row['level']} | {n} | {rho} | {p} | {q} | {dir_ok} |")

nb14_section = f'''
---

## Exploratory Site-Level Validation — ENIGMA MAG Geochemistry Discovery (Notebook 14)

**Analysis:** Geochemistry table discovery + Spearman ρ (140-KO density vs well metal concentration).  
**Geochemistry source selected:** {best_table} ({n_wells_geo} wells matched).  
**MAGs matched:** {n_mags_geo}/{len(mag_wells)}.  
**Pre-specified direction:** ρ > 0 (positive — high-metal wells → higher KO density).  
**Note:** Exploratory. Cannot bear on H1. Isolate genomes (n=2,925) excluded — no sample_id.

| Metal | Level | n | ρ | p_raw | p_FDR | Dir consistent |
|-------|-------|---|---|-------|-------|----------------|
{chr(10).join(table_rows)}
'''

marker = 'Exploratory Site-Level Validation — ENIGMA MAG Geochemistry Discovery'
if marker not in content:
    with open(interp_path, 'a') as f:
        f.write(nb14_section)
    print(f'Appended NB14 section to {interp_path}')
else:
    print('NB14 section already present — skipping.')

Appended NB14 section to /home/hmacgregor/BERIL-research-observatory/projects/comprehensive_metal_ecology/INTERPRETATION_TABLE.md
